# Simulate League (autoregressive Monte Carlo)

This notebook predicts the remaining fixtures from the first week where results are missing in `scores_with_venue.csv` and then autoregressively updates pooled parameters after each full week. It runs multiple Monte Carlo runs to estimate the probability of each final rank.

Ranking tie-break: points -> head-to-head points among tied teams -> head-to-head goal difference among tied teams; if still tied, an error is raised.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA = Path("data/super_league")
SCORES_CSV = DATA / "scores_with_venue.csv"
TEAMS_YAML = DATA / "teams.yaml"

N_RUNS = 100000
SEED = 12345

# Fixed venue multiplier (host lambda gets this applied)
HOME_LAMBDA_MULT = 1.08

In [3]:
def load_idx_to_name(yaml_path: Path) -> dict[int, str]:
    out: dict[int, str] = {}
    for raw in yaml_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        name, _, rest = line.partition(":")
        out[int(rest.strip())] = name.strip()
    return out


def parse_opponent_idx(cell: str) -> int | None:
    s = str(cell).strip()
    m = re.match(r"^([0-9]+):", s)
    return int(m.group(1)) if m else None


def parse_row_team_venue(cell: str) -> str | None:
    # `cell` format: `opponent_idx:H:<gf>-<ga>` or `opponent_idx:A:<gf>-<ga>`
    s = str(cell).strip()
    m = re.match(r"^[0-9]+:([HA]):", s)
    return m.group(1) if m else None


def parse_goals(cell: str) -> tuple[int, int] | None:
    # Returns (gf, ga) from the perspective of the row-team.
    s = str(cell).strip()
    if "?" in s:
        return None
    m = re.match(r"^[0-9]+:[HA]:([0-9]+)-([0-9]+)$", s)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def poisson_sample_score(rng: np.random.Generator, lam_home: float, lam_away: float) -> tuple[int, int]:
    lam_home = max(float(lam_home), 0.0)
    lam_away = max(float(lam_away), 0.0)
    return int(rng.poisson(lam_home)), int(rng.poisson(lam_away))


def resolve_bucket_tiebreak(matches: list[tuple[int, int, int, int]], bucket: list[int]) -> list[int]:
    """Tie-break points within bucket, then head-to-head GD and goals scored.
    Falls back to deterministic ordering by team index to avoid Monte Carlo crashes."""
    if len(bucket) == 1:
        return bucket

    bucket_set = set(bucket)
    # 1) head-to-head points
    head_points = {t: 0 for t in bucket}
    for i, j, gi, gj in matches:
        if i in bucket_set and j in bucket_set:
            if gi > gj:
                head_points[i] += 3
            elif gi < gj:
                head_points[j] += 3
            else:
                head_points[i] += 1
                head_points[j] += 1

    ordered: list[int] = []
    for hp in sorted({head_points[t] for t in bucket}, reverse=True):
        sub = [t for t in bucket if head_points[t] == hp]
        if len(sub) == 1:
            ordered.extend(sub)
            continue

        # 2) head-to-head goal difference within this subgroup
        sub_set = set(sub)
        gd = {t: 0 for t in sub}
        for i, j, gi, gj in matches:
            if i in sub_set and j in sub_set:
                gd[i] += gi - gj
                gd[j] += gj - gi

        gd_vals = {t: gd[t] for t in sub}
        unique_gd = sorted(set(gd_vals.values()), reverse=True)
        for g in unique_gd:
            sub2 = [t for t in sub if gd_vals[t] == g]
            if len(sub2) == 1:
                ordered.extend(sub2)
                continue

            # 3) head-to-head goals scored within this still-tied subgroup
            sub2_set = set(sub2)
            gf = {t: 0 for t in sub2}
            for i, j, gi, gj in matches:
                if i in sub2_set and j in sub2_set:
                    gf[i] += gi
                    gf[j] += gj

            unique_gf = sorted({gf[t] for t in sub2}, reverse=True)
            for gf_val in unique_gf:
                sub3 = [t for t in sub2 if gf[t] == gf_val]
                if len(sub3) == 1:
                    ordered.extend(sub3)
                else:
                    # Final fallback: deterministic order by team index
                    ordered.extend(sorted(sub3))

    return ordered


def rank_table(points: np.ndarray, matches: list[tuple[int, int, int, int]], n_teams: int) -> list[int]:
    """Return ordered teams from rank 1 to rank n_teams."""
    teams = list(range(n_teams))
    ordered: list[int] = []
    unique_pts = sorted({int(points[t]) for t in teams}, reverse=True)
    for pt in unique_pts:
        bucket = [t for t in teams if int(points[t]) == pt]
        if len(bucket) == 1:
            ordered.extend(bucket)
        else:
            ordered.extend(resolve_bucket_tiebreak(matches, bucket))
    return ordered

In [4]:
df = pd.read_csv(SCORES_CSV)
idx_to_name = load_idx_to_name(TEAMS_YAML)

team_idxs = sorted(int(x) for x in df["team_idx"].unique())
n_teams = len(team_idxs)

week_cols = [c for c in df.columns if c.startswith("w")]
week_nums = sorted(int(c[1:]) for c in week_cols)
week_cols_sorted = [f"w{w}" for w in week_nums]

# First week with any missing scoreline.
START_WEEK = 26
for wk in week_cols_sorted:
    if df[wk].astype(str).str.contains("?", regex=False).any():
        START_WEEK = wk
        break
if START_WEEK is None:
    raise ValueError("No missing scorelines found; nothing to simulate.")

print(f"Detected START_WEEK={START_WEEK}, last week={week_cols_sorted[-1]}")

# Fixtures: per week list of (low, high, host, guest, venue_low, cell_low)
fixtures_by_week: dict[str, list[tuple[int, int, int, int, str, str]]] = {}
for wk in week_cols_sorted:
    fixtures: list[tuple[int, int, int, int, str, str]] = []
    for _, row in df.iterrows():
        i = int(row["team_idx"])
        cell = str(row[wk]).strip()
        j = parse_opponent_idx(cell)
        if j is None:
            continue
        if i < j:
            venue_low = parse_row_team_venue(cell)
            if venue_low not in {"H", "A"}:
                continue
            low, high = i, j
            # host/guest based on the row team's venue in the low-row cell
            if venue_low == "H":
                host, guest = low, high
            else:
                host, guest = high, low
            fixtures.append((low, high, host, guest, venue_low, cell))
    fixtures.sort(key=lambda t: (t[0], t[1]))
    fixtures_by_week[wk] = fixtures

rank_counts = np.zeros((n_teams, n_teams), dtype=int)  # [team, rank_index]

last_week = week_cols_sorted[-1]
start_pos = week_cols_sorted.index(START_WEEK)

for run_id in range(N_RUNS):
    rng = np.random.default_rng(SEED + run_id)

    # Pooled parameters from realized matches so far.
    gf_sum = np.zeros(n_teams, dtype=float)
    gf_n = np.zeros(n_teams, dtype=float)
    ga_sum = np.zeros(n_teams, dtype=float)
    ga_n = np.zeros(n_teams, dtype=float)

    points = np.zeros(n_teams, dtype=int)
    matches: list[tuple[int, int, int, int]] = []  # (i, j, goals_i, goals_j)

    # Initialize with all weeks strictly before START_WEEK (assumed fully played)
    for wk in week_cols_sorted[:start_pos]:
        for low, high, host, guest, venue_low, cell_low in fixtures_by_week[wk]:
            actual = parse_goals(cell_low)
            if actual is None:
                raise ValueError(f"Run {run_id}: Unexpected missing game before START_WEEK in week={wk}, low={low}, high={high}")
            gf_low, ga_low = actual
            if venue_low == "H":
                host_goals, guest_goals = gf_low, ga_low
            else:
                host_goals, guest_goals = ga_low, gf_low

            # Points
            if host_goals > guest_goals:
                points[host] += 3
            elif host_goals < guest_goals:
                points[guest] += 3
            else:
                points[host] += 1
                points[guest] += 1

            # Parameters update
            gf_sum[host] += host_goals
            gf_n[host] += 1
            ga_sum[host] += guest_goals
            ga_n[host] += 1

            gf_sum[guest] += guest_goals
            gf_n[guest] += 1
            ga_sum[guest] += host_goals
            ga_n[guest] += 1

            matches.append((host, guest, host_goals, guest_goals))

    # Simulate START_WEEK..last_week
    for wk in week_cols_sorted[start_pos:]:
        # Compute pooled means once per full week.
        total_mu = gf_sum.sum() / gf_n.sum() if gf_n.sum() > 0 else float("nan")
        gf_mean = np.divide(gf_sum, gf_n, out=np.full_like(gf_sum, np.nan), where=gf_n > 0)
        ga_mean = np.divide(ga_sum, ga_n, out=np.full_like(ga_sum, np.nan), where=ga_n > 0)

        if not np.isfinite(total_mu) or total_mu <= 0:
            raise ValueError(f"Run {run_id}: total_mu invalid at week={wk} (need at least one played match before START_WEEK)")

        for low, high, host, guest, venue_low, cell_low in fixtures_by_week[wk]:
            actual = parse_goals(cell_low)
            if actual is not None:
                gf_low, ga_low = actual
                if venue_low == "H":
                    host_goals, guest_goals = gf_low, ga_low
                else:
                    host_goals, guest_goals = ga_low, gf_low
            else:
                s_host = float(gf_mean[host])
                c_guest = float(ga_mean[guest])
                s_guest = float(gf_mean[guest])
                c_host = float(ga_mean[host])

                if not (np.isfinite(s_host) and np.isfinite(c_guest) and np.isfinite(s_guest) and np.isfinite(c_host)):
                    raise ValueError(f"Run {run_id}: Missing mean stats for host={host}, guest={guest} at week={wk}")

                lam_host = s_host * (c_guest / total_mu) * HOME_LAMBDA_MULT
                lam_guest = s_guest * (c_host / total_mu)
                host_goals, guest_goals = poisson_sample_score(rng, lam_host, lam_guest)

            # Points
            if host_goals > guest_goals:
                points[host] += 3
            elif host_goals < guest_goals:
                points[guest] += 3
            else:
                points[host] += 1
                points[guest] += 1

            # Parameters update (for next weeks)
            gf_sum[host] += host_goals
            gf_n[host] += 1
            ga_sum[host] += guest_goals
            ga_n[host] += 1

            gf_sum[guest] += guest_goals
            gf_n[guest] += 1
            ga_sum[guest] += host_goals
            ga_n[guest] += 1

            matches.append((host, guest, host_goals, guest_goals))

        # end of week

    # Final ranking
    ranking = rank_table(points, matches, n_teams)
    for rank_pos, team in enumerate(ranking, start=1):
        rank_counts[team, rank_pos - 1] += 1


valid_runs = N_RUNS
rank_probs = rank_counts / valid_runs
rank_prob_df = pd.DataFrame(
    {
        "team_idx": list(range(n_teams)),
        "team": [idx_to_name[i] for i in range(n_teams)],
        **{f"P(rank={r+1})": rank_probs[:, r] for r in range(n_teams)},
    }
)

print("--- Rank probabilities ---")
rank_prob_df

# Compact summary
rank_cols = [f"P(rank={r+1})" for r in range(n_teams)]
p_champ = rank_prob_df["P(rank=1)"]
p_rank2 = rank_prob_df["P(rank=2)"]
p_rank3 = rank_prob_df["P(rank=3)"]
expected_rank = []
for i in range(n_teams):
    probs = rank_prob_df.loc[i, rank_cols].to_numpy(dtype=float)
    exp = float(np.sum((np.arange(1, n_teams + 1)) * probs))
    expected_rank.append(exp)

summary = pd.DataFrame(
    {
        "team": rank_prob_df["team"],
        "P(rank=1)": p_champ,
        "P(rank=2)": p_rank2,
        "P(rank=3)": p_rank3,
        "expected_rank": expected_rank,
    }
).sort_values(["P(rank=1)", "expected_rank"], ascending=[False, True])

print(f"Monte Carlo runs: {valid_runs}")
summary


Detected START_WEEK=w27, last week=w34
--- Rank probabilities ---
Monte Carlo runs: 100000


,team,P(rank=1),P(rank=2),P(rank=3),expected_rank
1,Galatasaray,0.92460,0.06372,0.01163,1.08713
14,Fenerbahce,0.06004,0.68796,0.23051,2.21364
10,Trabzonspor,0.01534,0.23092,0.63693,2.85801
17,Besiktas,0.00002,0.01740,0.11989,3.90446
7,Goztepe,0.00000,0.00000,0.00094,5.35759
13,Istanbul Basaksehir,0.00000,0.00000,0.00010,5.66339
2,Samsunspor,0.00000,0.00000,0.00000,8.35048
15,Alanyaspor,0.00000,0.00000,0.00000,9.07359
6,Rizespor,0.00000,0.00000,0.00000,9.51176
0,Gazisehir Gaziantep,0.00000,0.00000,0.00000,9.60979
